In [1]:
import os
os.environ["MKL_NUM_THREADS"] = "5"
os.environ["OPENBLAS_NUM_THREADS"] = "5"
os.environ["NUMEXPR_NUM_THREADS"] = "5"

# Example GLM standard

In [2]:
from importlib.resources import files
import pandas as pd
import numpy as np
from lepto.standard.model.linear_model import GLMDiff, transform_json_to_df
from lepto.gui.framework.utils import plot_distribution 

## Data import

In [3]:
path = files("lepto.data") / "sample_standard.csv"
df = pd.read_csv(path)

In [4]:
X = df[["age", "region", "segment"]]
y = df["y"].values
w = df["exposure"].values

In [5]:
plot_distribution(df['age'], sample_weights=None, var_name="", nbins=20)

## Model (simple GLM)

In [6]:
glm = GLMDiff(
    family="poisson",      
    tweedie_power=1.5,       
    fit_intercept=True,
    nbins=10,                
    lam=100,
    penalty_choice=None,
    offset_betas=None,
    sparse_output=True)

glm.fit(
    X=X,
    y=y,
    sample_weight=w
)

pred = glm.predict(X)
print(glm.score(X, y, w))
glm.summary

0.6702882010460411


{'intercept': np.float64(-1.6656524717123806),
 'coefficients': {'age': {'(27.0824, 31.6674]': np.float64(0.65285),
   '(31.6674, 34.7937]': np.float64(1.03912),
   '(34.7937, 37.5565]': np.float64(1.27688),
   '(37.5565, 39.9829]': np.float64(1.55993),
   '(39.9829, 42.5231]': np.float64(1.80509),
   '(42.5231, 45.2502]': np.float64(2.07639),
   '(45.2502, 48.4574]': np.float64(2.36563),
   '(48.4574, 53.0482]': np.float64(2.73385),
   '(53.0482, 81.5124]': np.float64(3.54995)},
  'segment': {1.0: np.float64(0.48831)},
  'region': {'north': np.float64(-0.01108),
   'south': np.float64(0.18276),
   'west': np.float64(0.00877)}},
 'link': 'log',
 'version': '0.1.0'}

In [7]:
glm.plot(X, y, w, "age")

In [9]:
# Model to excel format
transform_json_to_df(glm.summary)

,col0,col1,col2,col3,col4,col5,col6,col7,col8
0,Base,,-1.664916,,,,,,
1,,,,,,,,,
2,,,,,,,,,
3,age,,,segment,,,region,,
4,,,,,,,,,
5,age,,,segment,,,region,,
6,"(27.0824, 31.6674]",0.65212,,1.0,0.48829,,north,-0.01115,
7,"(31.6674, 34.7937]",1.03801,,,,,south,0.18278,
8,"(34.7937, 37.5565]",1.27649,,,,,west,0.00891,
9,"(37.5565, 39.9829]",1.55854,,,,,,,


## Model penalty

In [10]:
penalty = {'age': {'penalty':'continuous'}, 'segment': {'penalty':'categorical'}}

glm = GLMDiff(
    family="poisson",      
    tweedie_power=1.5,       
    fit_intercept=True,
    nbins=10,                
    lam=1e3,
    penalty_choice=penalty,
    offset_betas=None)

glm.fit(
    X=X,
    y=y,
    sample_weight=w
)

pred = glm.predict(X)
glm.summary

{'intercept': np.float64(-1.1108384358303938),
 'coefficients': {'age': {'(27.0824, 31.6674]': np.float64(0.22281),
   '(31.6674, 34.7937]': np.float64(0.50311),
   '(34.7937, 37.5565]': np.float64(0.76617),
   '(37.5565, 39.9829]': np.float64(1.03533),
   '(39.9829, 42.5231]': np.float64(1.29595),
   '(42.5231, 45.2502]': np.float64(1.57059),
   '(45.2502, 48.4574]': np.float64(1.87478),
   '(48.4574, 53.0482]': np.float64(2.27344),
   '(53.0482, 81.5124]': np.float64(2.96817)},
  'segment': {1.0: np.float64(0.42561)},
  'region': {'north': np.float64(-0.00957),
   'south': np.float64(0.18408),
   'west': np.float64(0.01055)}},
 'link': 'log',
 'version': '0.0.0+local'}

In [11]:
glm.model.betas

array([ 0.22281   ,  0.50311   ,  0.76617   ,  1.03533   ,  1.29595   ,
        1.57059   ,  1.87478   ,  2.27344   ,  2.96817   ,  0.42561   ,
       -0.00957   ,  0.18408   ,  0.01055   , -1.11083844])

## Model offset

In [12]:
# Offset model to force one coefficient to be at specific values. You need to know the position of the coefficient in the model. It can be done via glm.model.betas
offset_betas = np.array([ np.nan  ,  np.nan   ,  np.nan   ,  np.nan   ,  np.nan   ,
        np.nan  ,  np.nan  ,  np.nan  ,  np.nan   , np.nan   ,
        1.5   ,  np.nan   ,  np.nan   , np.nan])

glm = GLMDiff(
    family="poisson",      
    tweedie_power=1.5,       
    fit_intercept=True,
    nbins=10,                
    lam=1e8,
    penalty_choice=None,
    offset_betas=offset_betas)

glm.fit(
    X=X,
    y=y,
    sample_weight=w
)

pred = glm.predict(X)
glm.summary

{'intercept': np.float64(-2.687215362815512),
 'coefficients': {'age': {'(27.0824, 31.6674]': np.float64(0.66258),
   '(31.6674, 34.7937]': np.float64(1.04545),
   '(34.7937, 37.5565]': np.float64(1.26986),
   '(37.5565, 39.9829]': np.float64(1.56575),
   '(39.9829, 42.5231]': np.float64(1.81592),
   '(42.5231, 45.2502]': np.float64(2.08168),
   '(45.2502, 48.4574]': np.float64(2.37012),
   '(48.4574, 53.0482]': np.float64(2.76502),
   '(53.0482, 81.5124]': np.float64(3.54423)},
  'segment': {1.0: np.float64(0.50549)},
  'region': {'north': np.float64(1.5),
   'south': np.float64(1.19004),
   'west': np.float64(1.0161)}},
 'link': 'log',
 'version': '0.0.0+local'}